# Tutorial 02 — Model 2: Dual-Plume Scenario

This notebook covers the more demanding two-plume test case used in the HWSI paper.

**Scenario summary**

| | Plume A | Plume B |
|---|---|---|
| Resistivity | 10 Ωm (strongly conductive) | 55 → 30 Ωm (weakly conductive) |
| Centre | (−4 m, −1 m) | (+5 m, 0 m) |
| Radius growth | 0.3 → 2.5 m | 0.2 → 1.8 m |
| Boundary shape | Fourier-perturbed ellipse | Fourier-perturbed ellipse |
| Contrast vs background | 60:1 | ~20:1 |

The co-existence of two anomalies with markedly different contrast ratios is the key challenge: a uniform temporal coupling calibrated to Plume A will suppress Plume B's weaker signal, and vice versa.

---

**Dependencies**: pyGIMLi 1.5.5, NumPy ≥ 1.23, SciPy ≥ 1.9, Matplotlib ≥ 3.6  
See `README.md` for installation instructions.

## 0. Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm

import pygimli.meshtools as mt
from pygimli.physics import ert

from hwsi import (
    HWSI_Inversion,
    run_independent,
    run_difference,
    run_4d,
    interp_to_grid,
    build_valid_mask,
    calc_errors,
)

%matplotlib inline
print('All imports successful.')

## 1. Mesh Construction

Same mesh setup as Model 1 — two meshes with different topology to avoid the inverse crime.

In [ ]:
scheme = ert.createData(elecs=np.linspace(-12, 12, 50), schemeName='dd')

world = mt.createWorld(start=[-25, -15], end=[25, 0], worldMarker=True)
for p in scheme.sensorPositions():
    world.createNode(p)
fwd_mesh = mt.createMesh(world, quality=34.5, area=0.2)

inv_domain = mt.createRectangle(start=[-12, -6], end=[12, 0])
inv_mesh   = mt.createMesh(inv_domain, quality=34.5, area=0.1)

grid_x, grid_z = np.mgrid[-12:12:300j, -6:0:150j]

print(f'Forward mesh : {fwd_mesh.cellCount():,} cells')
print(f'Inversion mesh: {inv_mesh.cellCount():,} cells')

## 2. True Resistivity Model

Both plumes have Fourier-perturbed elliptical boundaries.  The perturbation shape is fixed by a random seed so results are fully reproducible.

In [ ]:
BG_RHO = 600.0

PLUME_A = dict(
    cx=-4.0, cz=-1.0, rho=10.0,
    r_min=0.3, r_max=2.5, ax=1.0, az=1.4,
    n_modes=4, amp=0.25, phase_offset=0.0,
)
PLUME_B = dict(
    cx=+5.0, cz=0.0, rho_start=55.0, rho_end=30.0,
    r_min=0.2, r_max=1.8, ax=1.0, az=1.2,
    n_modes=4, amp=0.20, phase_offset=1.1,
)

# Fixed random perturbation weights (seed=123 matches paper)
_RNG    = np.random.default_rng(seed=123)
_PERT_A = _RNG.uniform(-1, 1, PLUME_A['n_modes'])
_PERT_B = _RNG.uniform(-1, 1, PLUME_B['n_modes'])


def _ellipse_r(theta, r_base, ax, az, amp, n_modes, pert, phase):
    """Radius of a Fourier-perturbed ellipse at angle theta."""
    theta = np.atleast_1d(np.asarray(theta, dtype=float))
    r_ell = r_base / np.sqrt((np.cos(theta)/ax)**2 + (np.sin(theta)/az)**2)
    pert_v = sum(
        pert[k-1] * np.sin(k*theta + phase + k*0.5)
        for k in range(1, n_modes+1)
    )
    r_bnd = np.maximum(r_ell * (1.0 + amp*pert_v), r_ell*0.3)
    return float(r_bnd[0]) if r_bnd.size == 1 else r_bnd


def _radii(t_hour):
    prog  = np.clip(t_hour/24.0, 0.0, 1.0)
    r_A   = PLUME_A['r_min'] + (PLUME_A['r_max'] - PLUME_A['r_min']) * prog
    r_B   = PLUME_B['r_min'] + (PLUME_B['r_max'] - PLUME_B['r_min']) * prog
    rho_B = PLUME_B['rho_start'] + (PLUME_B['rho_end'] - PLUME_B['rho_start']) * prog
    return r_A, r_B, rho_B


def get_true_rho_grid(gx, gz, t_hour):
    """True resistivity on the regular grid at time t_hour."""
    rho = np.full(gx.shape, BG_RHO)
    if t_hour <= 0:
        return rho
    r_A, r_B, rho_B = _radii(t_hour)
    sub = gz <= 0
    for plume, r, val, pert in [
        (PLUME_B, r_B, rho_B,        _PERT_B),
        (PLUME_A, r_A, PLUME_A['rho'], _PERT_A),
    ]:
        dx    = gx - plume['cx']; dz = gz - plume['cz']
        r_bnd = _ellipse_r(
            np.arctan2(dz, dx), r,
            plume['ax'], plume['az'],
            plume['amp'], plume['n_modes'],
            pert, plume['phase_offset'],
        )
        rho[sub & (np.sqrt(dx**2 + dz**2) <= r_bnd)] = val
    return rho


def get_true_rho_point(x, z, t_hour):
    """True resistivity at a single point and time."""
    if z > 0:
        return BG_RHO
    if t_hour <= 0:
        return BG_RHO
    r_A, r_B, rho_B = _radii(t_hour)
    for plume, r, val, pert in [
        (PLUME_B, r_B, rho_B,        _PERT_B),
        (PLUME_A, r_A, PLUME_A['rho'], _PERT_A),
    ]:
        dx = x - plume['cx']; dz = z - plume['cz']
        r_bnd = _ellipse_r(
            np.arctan2(dz, dx), r,
            plume['ax'], plume['az'],
            plume['amp'], plume['n_modes'],
            pert, plume['phase_offset'],
        )
        if np.sqrt(dx**2 + dz**2) <= r_bnd:
            return val
    return BG_RHO


# Visualise the true model at t = 24 h
tg_24 = get_true_rho_grid(grid_x, grid_z, 24)
fig, ax = plt.subplots(figsize=(9, 3))
im = ax.imshow(
    np.log10(tg_24).T, origin='lower', cmap='jet',
    vmin=0, vmax=4, extent=[-12, 12, -6, 0], aspect='equal',
)
plt.colorbar(im, ax=ax, label='log10(rho) [Ωm]')
ax.set_title('True model at t = 24 h — Plume A (left) and Plume B (right)')
ax.set_xlabel('Distance (m)'); ax.set_ylabel('Depth (m)')
plt.tight_layout()
plt.show()

## 3. Synthetic Data Generation

In [ ]:
NOISE_LEVEL = 0.03
DATA_ERROR  = 0.05
times = np.arange(0, 25, 4)

data_list, true_grids = [], []

print('Generating synthetic datasets...')
for t in times:
    rho_vec = np.array([
        get_true_rho_point(c.center().x(), c.center().y(), t)
        for c in fwd_mesh.cells()
    ])
    data = ert.simulate(
        mesh=fwd_mesh, scheme=scheme, res=rho_vec,
        noiseLevel=NOISE_LEVEL, seed=123, verbose=False,
    )
    data['rhoa'] = np.maximum(np.abs(data['rhoa']), 0.1)
    data.set('err', np.full(data.size(), DATA_ERROR))
    data_list.append(data)

    tg = get_true_rho_grid(grid_x, grid_z, t)
    true_grids.append(tg)
    n_A = int(np.sum(tg < 15))
    n_B = int(np.sum((tg >= 15) & (tg < 200)))
    print(f'  T={t:2d}h | Plume A cells: {n_A:5d} | Plume B cells: {n_B:5d}')

print('Done.')

## 4. Running the Inversions

In [ ]:
LAMBDA_S = 20.0
LAMBDA_T = 20.0

results_indep = run_independent(inv_mesh, data_list, lam=LAMBDA_S)

In [ ]:
results_diff = run_difference(inv_mesh, data_list, lam=LAMBDA_S)

In [ ]:
results_4d = run_4d(inv_mesh, data_list, lam=LAMBDA_S, scalef=1.0)

In [ ]:
hwsi = HWSI_Inversion(
    inv_mesh, data_list,
    lambda_s=LAMBDA_S, lambda_t=LAMBDA_T,
    epsilon=5e-3, alpha_max=0.65,
    max_iter=15, verbose=True,
)
results_hwsi = hwsi.run()

## 5. Error Evaluation

In [ ]:
results_dict = {
    'Independent':   results_indep,
    'Difference':    results_diff,
    '4D L2-Coupled': results_4d,
    'HWSI':          results_hwsi,
}

valid_mask = build_valid_mask(inv_mesh, grid_x, grid_z)
errors     = calc_errors(
    times, true_grids, results_dict,
    grid_x, grid_z, inv_mesh, mask=valid_mask,
)

baseline = np.nanmean(errors['Independent']['nrmse'])
print(f"{'Method':<20} {'Mean nRMSE':>11} {'Std':>7} {'vs Indep':>10}")
print('-' * 52)
for mn in results_dict:
    avg = np.nanmean(errors[mn]['nrmse'][1:])
    std = np.nanstd(errors[mn]['nrmse'][1:])
    print(f"{mn:<20} {avg:>10.2f}% {std:>6.2f}%  {baseline - avg:>+8.2f} pp")

## 6. nRMSE Plot

In [ ]:
METHOD_COLORS = {
    'Independent':   '#1f77b4',
    'Difference':    '#ff7f0e',
    '4D L2-Coupled': '#d62728',
    'HWSI':          '#2ca02c',
}

fig, ax = plt.subplots(figsize=(9, 5))
xt = range(1, len(times))
for mn, vals in errors.items():
    ax.plot(xt, vals['nrmse'][1:], lw=2.2,
            label=mn, color=METHOD_COLORS[mn])
ax.set_xticks(list(xt))
ax.set_xticklabels([f'T={t}h' for t in times[1:]])
ax.set_ylabel('nRMSE (%)')
ax.set_ylim(20, 100)
ax.grid(alpha=0.3, ls='--')
ax.legend()
ax.set_title('Model 2 — nRMSE trajectories')
plt.tight_layout()
plt.show()

## 7. Why Uniform Coupling Fails Here

The cell below demonstrates the key finding: at the final timestep (t = 24 h), both Difference and 4D L2-Coupled finish **worse** than the no-coupling baseline.  HWSI is the only method to improve on Independent inversion across the full sequence.

In [ ]:
ti_last = len(times) - 1  # t = 24 h
print(f'nRMSE at t = {times[ti_last]} h')
print('-' * 35)
for mn in results_dict:
    val = errors[mn]['nrmse'][ti_last]
    marker = ' <-- best' if mn == 'HWSI' else ''
    print(f'  {mn:<20} {val:>6.2f}%{marker}')

## 8. Side-by-Side Comparison at t = 24 h

In [ ]:
norm = Normalize(vmin=0, vmax=4)
fig, axes = plt.subplots(1, 4, figsize=(22, 4))

for ax, (mn, res) in zip(axes, results_dict.items()):
    gd = interp_to_grid(inv_mesh, res[ti_last], grid_x, grid_z)
    ax.imshow(
        np.log10(np.clip(gd, 1, 1e4)).T,
        origin='lower', cmap='jet', norm=norm,
        extent=[-12, 12, -6, 0], aspect='equal',
    )
    ax.set_title(f'{mn}\nnRMSE = {errors[mn]["nrmse"][ti_last]:.1f}%')
    ax.set_xlabel('Distance (m)')
axes[0].set_ylabel('Depth (m)')
plt.suptitle('Inverted resistivity at t = 24 h', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 9. Temporal Change Maps at the Final Transition (20 h → 24 h)

In [ ]:
ti_prev = list(times).index(20)
ti_curr = list(times).index(24)

fig, axes = plt.subplots(1, 4, figsize=(22, 4))
cmap_jw = plt.cm.RdBu_r

for ax, (mn, res) in zip(axes, results_dict.items()):
    rp  = np.clip(interp_to_grid(inv_mesh, res[ti_prev], grid_x, grid_z), 1, 1e4)
    rc  = np.clip(interp_to_grid(inv_mesh, res[ti_curr], grid_x, grid_z), 1, 1e4)
    chg = np.log10(rc / rp)
    chg_masked = np.where(np.abs(chg) < 0.12, np.nan, chg)
    ax.imshow(
        chg_masked.T, origin='lower', cmap=cmap_jw,
        vmin=-0.6, vmax=0.6,
        extent=[-12, 12, -6, 0], aspect='equal',
    )
    ax.set_facecolor('white')
    ax.set_title(f'{mn}\n20h → 24h')
    ax.set_xlabel('Distance (m)')
axes[0].set_ylabel('Depth (m)')
plt.suptitle(
    r'$\log_{10}(\rho_{24h} / \rho_{20h})$ — threshold 0.12',
    fontsize=13, y=1.02,
)
plt.tight_layout()
plt.show()

## 10. HWSI Convergence — Model 2

In [ ]:
hist  = hwsi.convergence_history
iters = [h['iteration']  for h in hist]
avgs  = [h['avg_change'] for h in hist]

fig, ax = plt.subplots(figsize=(5, 4))
ax.semilogy(iters, avgs, color='#2ca02c', lw=2.4)
ax.set_xlabel('Iteration')
ax.set_ylabel(r'Avg $||\Delta m||$')
ax.set_xticks(iters)
ax.grid(alpha=0.3)
ax.set_title('HWSI convergence — Model 2')
plt.tight_layout()
plt.show()
print(f'Converged at iteration {iters[-1]} — more iterations than Model 1')
print('(two weight fields evolving on different timescales)')

---

For high-resolution publication figures, run `models/model2_two_plume.py` directly.

For sensitivity analysis (regularisation grid, noise levels, α_max sweep), modify the parameter blocks in `models/model2_two_plume.py` and re-run — all four methods will be invoked automatically.